##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

1- ResNet50V2 achieved the highest accuracy.

ResNet50V2 fine-tuned accuracy: 91.62%

MobileNetV2 fine-tuned accuracy: 89.67%

This shows that the deeper residual architecture of ResNet50V2 provides stronger feature learning compared to MobileNetV2.

2- MobileNetV2 trained significantly faster.

Average training time per epoch:

Model	Time / Epoch
MobileNetV2	~82–115 sec
ResNet50V2	~209 sec

MobileNetV2 is almost 2× faster than ResNet50V2.

3- The performance difference can be explained by the network design.

ResNet50V2

Very deep network (190 layers)

Uses residual skip connections

Improves gradient flow during training

Learns more complex visual representations

Higher accuracy but slower training.

In [2]:
import time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TF version:", tf.__version__)
tf.random.set_seed(42)
np.random.seed(42)

2026-02-24 18:34:36.712329: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF version: 2.12.0


In [3]:
# ----------------------------
# 1) Load CIFAR-10
# ----------------------------
(num_classes, input_size) = (10, 96)  # MobileNetV2 works great at 96 or 128
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.squeeze()
y_test  = y_test.squeeze()

# Split train -> train/val
val_frac = 0.1
n_val = int(len(x_train) * val_frac)

x_val, y_val = x_train[:n_val], y_train[:n_val]
x_tr,  y_tr  = x_train[n_val:], y_train[n_val:]

print("Train:", x_tr.shape, y_tr.shape)
print("Val:  ", x_val.shape, y_val.shape)
print("Test: ", x_test.shape, y_test.shape)

Train: (45000, 32, 32, 3) (45000,)
Val:   (5000, 32, 32, 3) (5000,)
Test:  (10000, 32, 32, 3) (10000,)


In [4]:

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 64

# Optional: light augmentation (helps CIFAR-10 a lot)
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augment")

def preprocess_train(image, label):
    image = tf.image.resize(image, (input_size, input_size))
    image = tf.cast(image, tf.float32)
    image = augment(image)
    image = preprocess_input(image)  # expects float image, scales to MobileNetV2 range
    return image, label

def preprocess_eval(image, label):
    image = tf.image.resize(image, (input_size, input_size))
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((x_tr, y_tr)).shuffle(10000).map(
    preprocess_train, num_parallel_calls=AUTOTUNE
).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).map(
    preprocess_eval, num_parallel_calls=AUTOTUNE
).batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).map(
    preprocess_eval, num_parallel_calls=AUTOTUNE
).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [ ]:

base_model = keras.applications.MobileNetV2(
    input_shape=(input_size, input_size, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # freeze for phase 1

inputs = keras.Input(shape=(input_size, input_size, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="MobileNetV2_CIFAR10")


model.summary()

print("\nTotal params:", model.count_params())
print("Trainable params:", np.sum([np.prod(v.shape) for v in model.trainable_weights]))
print("Non-trainable params:", np.sum([np.prod(v.shape) for v in model.non_trainable_weights]))
print("Base trainable:", base_model.trainable)

Model: "MobileNetV2_CIFAR10"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 96, 96, 3)]       0         
                                                                 
 mobilenetv2_1.00_96 (Functi  (None, 3, 3, 1280)       2257984   
 onal)                                                           
                                                                 
 global_average_pooling2d (G  (None, 1280)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 10)                12810     
                                                                 
Total params: 2,270,794
Trainable params: 12,81

In [6]:

class TimeHistory(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.epoch_times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_times.append(time.time() - self._start)

time_cb = TimeHistory()

In [7]:
# ----------------------------
# 5) Phase 1: Train head only (frozen backbone)
# ----------------------------
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, verbose=1),
    time_cb
]

history_frozen = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

print("\nAvg sec/epoch (frozen):", np.mean(time_cb.epoch_times))

2026-02-24 18:34:55.119144: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype uint8 and shape [45000,32,32,3]
	 [[{{node Placeholder/_0}}]]
2026-02-24 18:34:55.119473: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype uint8 and shape [45000]
	 [[{{node Placeholder/_1}}]]


Epoch 1/10
703/704 [============================>.] - ETA: 0s - loss: 0.8279 - accuracy: 0.7203

2026-02-24 18:36:04.888432: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype uint8 and shape [5000]
	 [[{{node Placeholder/_1}}]]
2026-02-24 18:36:04.888647: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype uint8 and shape [5000]
	 [[{{node Placeholder/_1}}]]


704/704 [==============================] - 77s 107ms/step - loss: 0.8280 - accuracy: 0.7202 - val_loss: 0.5041 - val_accuracy: 0.8314 - lr: 0.0010
Epoch 2/10
704/704 [==============================] - 79s 112ms/step - loss: 0.6294 - accuracy: 0.7850 - val_loss: 0.4402 - val_accuracy: 0.8546 - lr: 0.0010
Epoch 3/10
704/704 [==============================] - 77s 109ms/step - loss: 0.5987 - accuracy: 0.7936 - val_loss: 0.4525 - val_accuracy: 0.8454 - lr: 0.0010
Epoch 4/10
703/704 [============================>.] - ETA: 0s - loss: 0.5896 - accuracy: 0.7978
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0003000000142492354.
704/704 [==============================] - 89s 126ms/step - loss: 0.5896 - accuracy: 0.7978 - val_loss: 0.4523 - val_accuracy: 0.8460 - lr: 0.0010
Epoch 5/10
704/704 [==============================] - 90s 127ms/step - loss: 0.5492 - accuracy: 0.8100 - val_loss: 0.4275 - val_accuracy: 0.8534 - lr: 3.0000e-04

Avg sec/epoch (frozen): 82.16135501861572


In [8]:
# Evaluate after Phase 1
test_loss_1, test_acc_1 = model.evaluate(test_ds, verbose=0)
print(f"Phase 1 (frozen) Test Accuracy: {test_acc_1:.4f}")

2026-02-24 18:42:21.461803: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype uint8 and shape [10000]
	 [[{{node Placeholder/_1}}]]
2026-02-24 18:42:21.463540: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_1' with dtype uint8 and shape [10000]
	 [[{{node Placeholder/_1}}]]


Phase 1 (frozen) Test Accuracy: 0.8436


In [9]:
# ----------------------------
# 6) Phase 2: Fine-tune (unfreeze last N layers)
# ----------------------------
base_model.trainable = True

# Freeze all layers except the last N layers
N = 40  
for layer in base_model.layers[:-N]:
    layer.trainable = False


for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

print("Base trainable:", base_model.trainable)
print("Trainable layers in base:", sum([l.trainable for l in base_model.layers]), "/", len(base_model.layers))


print("\nTrainable params:", np.sum([np.prod(v.shape) for v in model.trainable_weights]))
print("Non-trainable params:", np.sum([np.prod(v.shape) for v in model.non_trainable_weights]))

Base trainable: True
Trainable layers in base: 26 / 154

Trainable params: 1676170
Non-trainable params: 594624


In [10]:
# Re-compile with a LOWER LR for fine-tuning
time_cb_ft = TimeHistory()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_ft = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, verbose=1),
    time_cb_ft
]

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks_ft
)

print("\nAvg sec/epoch (fine-tune):", np.mean(time_cb_ft.epoch_times))

Epoch 1/10
704/704 [==============================] - 107s 149ms/step - loss: 0.5345 - accuracy: 0.8149 - val_loss: 0.4159 - val_accuracy: 0.8628 - lr: 1.0000e-05
Epoch 2/10
704/704 [==============================] - 133s 188ms/step - loss: 0.4686 - accuracy: 0.8377 - val_loss: 0.3640 - val_accuracy: 0.8774 - lr: 1.0000e-05
Epoch 3/10
704/704 [==============================] - 127s 180ms/step - loss: 0.4299 - accuracy: 0.8494 - val_loss: 0.3456 - val_accuracy: 0.8852 - lr: 1.0000e-05
Epoch 4/10
704/704 [==============================] - 106s 150ms/step - loss: 0.3966 - accuracy: 0.8606 - val_loss: 0.3462 - val_accuracy: 0.8844 - lr: 1.0000e-05
Epoch 5/10
704/704 [==============================] - 117s 167ms/step - loss: 0.3719 - accuracy: 0.8689 - val_loss: 0.3218 - val_accuracy: 0.8924 - lr: 1.0000e-05
Epoch 6/10
704/704 [==============================] - 114s 162ms/step - loss: 0.3505 - accuracy: 0.8772 - val_loss: 0.3106 - val_accuracy: 0.8964 - lr: 1.0000e-05
Epoch 7/10
704/704 [==

In [11]:
# Evaluate after Phase 2
test_loss_2, test_acc_2 = model.evaluate(test_ds, verbose=0)
print(f"Phase 2 (fine-tuned) Test Accuracy: {test_acc_2:.4f}")

Phase 2 (fine-tuned) Test Accuracy: 0.8967


In [12]:

results = {
    "MobileNetV2_frozen_test_acc": float(test_acc_1),
    "MobileNetV2_finetune_test_acc": float(test_acc_2),
    "MobileNetV2_frozen_sec_per_epoch": float(np.mean(time_cb.epoch_times)),
    "MobileNetV2_finetune_sec_per_epoch": float(np.mean(time_cb_ft.epoch_times)),
}

results

{'MobileNetV2_frozen_test_acc': 0.8435999751091003,
 'MobileNetV2_finetune_test_acc': 0.8967000246047974,
 'MobileNetV2_frozen_sec_per_epoch': 82.16135501861572,
 'MobileNetV2_finetune_sec_per_epoch': 115.50079145431519}